# Connect-4 DQN v2 — Improved Dueling Double DQN with Self-Play

### Key improvements over v1
| Issue in v1 | Fix in v2 |
|---|---|
| Only terminal transitions stored (1/game) | Every DQN decision stored (~15/game) |
| `list(deque)` sampling — O(n) per step | Pre-allocated numpy arrays — O(1) |
| No `@tf.function` → 28 s/episode | Compiled train step → <2 s/episode |
| Fixed synthetic opponents | Self-play: trains against snapshots of itself |
| 1,500 total game transitions | 20K+ games → 300K+ transitions |
| Basic CNN Q-network | Dueling DQN with residual block |
| No reward shaping | Dense rewards for threats/blocks |

**To run:** Runtime → Change runtime type → T4 GPU → Run All

In [ ]:
# Cell 1: Colab Setup
import os, sys, json, time, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import matplotlib.pyplot as plt

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
tf.get_logger().setLevel('ERROR')

# Allow GPU memory growth (prevents TF from grabbing all 15 GB upfront)
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus[0].name if gpus else 'None — using CPU (slower)')

# Clone the repo so we have the src/ modules and model files
REPO_URL = 'https://github.com/Stiles-Clements1/connect4-rl-arena.git'
REPO_DIR = Path('/content/connect4-rl-arena')
if not REPO_DIR.exists():
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    print('Repo cloned.')
else:
    print('Repo already present.')

sys.path.insert(0, str(REPO_DIR / 'src'))
for d in ['checkpoints', 'logs']:
    (REPO_DIR / d).mkdir(exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('Setup complete. Working dir:', REPO_DIR)

In [ ]:
# Cell 2: Hyperparameters
# ── All tunable values live here. Change only this cell. ────────────

# DQN algorithm
GAMMA           = 0.99      # discount factor
LR              = 3e-4      # Adam learning rate
BATCH_SIZE      = 256       # gradient batch (power-of-2 for GPU efficiency)
BUFFER_CAP      = 200_000   # replay buffer capacity (large = diverse experiences)
TARGET_SYNC_N   = 1_000     # copy online → target every N gradient steps
GRAD_CLIP       = 1.0

# Training schedule
NUM_EPISODES    = 400       # training episodes
GAMES_PER_EP    = 50        # games collected per episode
UPDATES_PER_GAME = 4        # gradient steps per game collected
WARMUP_EPS      = 5         # episodes of pure exploration before any training

# Exploration (epsilon-greedy)
EPS_START       = 1.0
EPS_END         = 0.05
EPS_DECAY       = 0.992     # multiplicative per episode; reaches ~0.05 at ep ~430

# Self-play
SELFPLAY_FREQ   = 25        # snapshot model into pool every N episodes
SELFPLAY_CAP    = 10        # max pool size (drop oldest when full)
SELFPLAY_FRAC   = 0.6       # fraction of games vs self-play opponents

# Reward shaping (intermediate dense signals)
R_WIN    =  1.0
R_LOSS   = -1.0
R_DRAW   =  0.05   # draws are slightly better than losses
R_THREAT =  0.03   # DQN created a new open 3-in-a-row
R_BLOCK  =  0.02   # DQN eliminated an opponent threat

# Logging / checkpointing
CKPT_EVERY  = 50
EVAL_GAMES  = 200   # games per evaluation run (greedy, no exploration)

TOTAL_GAMES = NUM_EPISODES * GAMES_PER_EP
print(f'Config: {NUM_EPISODES} episodes × {GAMES_PER_EP} games = {TOTAL_GAMES:,} total games')
print(f'Buffer: {BUFFER_CAP:,} cap  |  Batch: {BATCH_SIZE}  |  LR: {LR}')

In [ ]:
# Cell 3: Dueling DQN Architecture
#
# Research basis:
#   Dueling Networks (Wang et al. 2016) — separate V(s) and A(s,a) streams.
#   In Connect-4, many moves have similar values in mid-game positions.
#   Dueling prevents the network from conflating 'this state is bad'
#   with 'this specific action is bad', reducing Q-value overestimation.
#
#   Residual connections — deeper feature extraction without vanishing gradients.
#   The (3,3) kernel matches the connect-4 winning window width.

def build_dueling_dqn(input_shape=(6, 7, 2), n_actions=7, name='online'):
    """
    Input:  (6, 7, 2) board — ch0 = DQN's pieces, ch1 = opponent's pieces.
    Output: Q-values for each of the 7 columns.
    Q(s,a) = V(s) + [A(s,a) − mean_a A(s,a)]  (standard Dueling formulation)
    """
    inp = layers.Input(shape=input_shape, name='board')

    # Shared convolutional trunk
    x = layers.Conv2D(64,  (3, 3), padding='same', activation='relu')(inp)
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)

    # Residual block — lets gradient flow around the extra conv layer
    res = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    res = layers.Conv2D(128, (3, 3), padding='same')(res)
    x   = layers.Add()([x, res])
    x   = layers.Activation('relu')(x)

    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)

    # Value stream: V(s) — scalar estimate of how good the state is
    v = layers.Dense(256, activation='relu')(x)
    v = layers.Dense(1, name='value')(v)

    # Advantage stream: A(s, a) — how much better each action is vs average
    a = layers.Dense(256, activation='relu')(x)
    a = layers.Dense(n_actions, name='advantage')(a)

    # Combine streams (mean-centering A removes the identifiability issue)
    q = layers.Lambda(
        lambda va: va[0] + (va[1] - tf.reduce_mean(va[1], axis=1, keepdims=True)),
        name='q_values'
    )([v, a])

    return models.Model(inp, q, name=name)


online_net = build_dueling_dqn(name='online')
target_net = build_dueling_dqn(name='target')
target_net.set_weights(online_net.get_weights())
target_net.trainable = False   # target is never directly optimised

optimizer = optimizers.Adam(learning_rate=LR, clipnorm=GRAD_CLIP)

print(f'Dueling DQN — {online_net.count_params():,} parameters')
online_net.summary(line_length=80)

In [ ]:
# Cell 4: Numpy Replay Buffer
#
# v1 used a deque and called list(replay_buffer) before every sample —
# an O(n) allocation repeated ~4,000 times = ~200s wasted.
#
# Pre-allocated numpy arrays let us sample with integer fancy-indexing in O(1).
# Memory: 200K × 6×7×2 × float32 × 2 (s + s') ≈ 1.2 GB — fits Colab T4 (15 GB).

class ReplayBuffer:
    """Circular buffer backed by pre-allocated numpy arrays."""

    def __init__(self, capacity, state_shape=(6, 7, 2)):
        self.cap  = capacity
        self.ptr  = 0
        self.size = 0
        self.S  = np.zeros((capacity, *state_shape), dtype=np.float32)
        self.A  = np.zeros(capacity,                 dtype=np.int32)
        self.R  = np.zeros(capacity,                 dtype=np.float32)
        self.S2 = np.zeros((capacity, *state_shape), dtype=np.float32)
        self.D  = np.zeros(capacity,                 dtype=np.float32)

    def push(self, s, a, r, s2, done):
        i = self.ptr
        self.S[i]  = s
        self.A[i]  = a
        self.R[i]  = r
        self.S2[i] = s2
        self.D[i]  = float(done)
        self.ptr  = (self.ptr + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, n):
        idx = np.random.randint(0, self.size, n)
        return (
            tf.constant(self.S[idx]),
            tf.constant(self.A[idx]),
            tf.constant(self.R[idx]),
            tf.constant(self.S2[idx]),
            tf.constant(self.D[idx]),
        )

    def __len__(self):
        return self.size


buf = ReplayBuffer(BUFFER_CAP)
mb  = BUFFER_CAP * 6 * 7 * 2 * 4 * 2 / 1e6
print(f'Replay buffer: {BUFFER_CAP:,} transitions  (~{mb:.0f} MB)')

In [ ]:
# Cell 5: Game Engine + Reward Shaping

# ── Board operations ─────────────────────────────────────────────────

def legal_cols(board):
    return [c for c in range(7) if board[0, c] == 0]

def drop_piece(board, col, player):
    """Return new board with player's piece dropped in col (lowest empty row)."""
    b = board.copy()
    for row in range(5, -1, -1):
        if b[row, col] == 0:
            b[row, col] = player
            return b
    raise ValueError(f'Column {col} is full')

def wins(board, player):
    """Vectorised four-in-a-row check — avoids slow Python loops."""
    b = (board == player)
    if (b[:, :-3] & b[:, 1:-2] & b[:, 2:-1] & b[:, 3:]).any():   return True  # horizontal
    if (b[:-3, :] & b[1:-2, :] & b[2:-1, :] & b[3:, :]).any():   return True  # vertical
    if (b[:-3,:-3] & b[1:-2,1:-2] & b[2:-1,2:-1] & b[3:,3:]).any(): return True  # diag \
    if (b[:-3, 3:] & b[1:-2,2:-1] & b[2:-1,1:-2] & b[3:,:-3]).any(): return True  # diag /
    return False

def encode(board, player):
    """6×7×2 float32 — ch0 = my pieces, ch1 = opponent's pieces."""
    s = np.zeros((6, 7, 2), dtype=np.float32)
    s[:, :, 0] = (board ==  player)
    s[:, :, 1] = (board == -player)
    return s


# ── Reward shaping ───────────────────────────────────────────────────

def _count_open_threes(board, player):
    """Count windows of 3 player pieces + 1 empty (immediate-threat patterns)."""
    n = 0
    for r in range(6):
        for c in range(4):
            w = board[r, c:c+4]
            if (w == player).sum() == 3 and (w == 0).sum() == 1:
                n += 1
    for r in range(3):
        for c in range(7):
            w = board[r:r+4, c]
            if (w == player).sum() == 3 and (w == 0).sum() == 1:
                n += 1
    return n

def shaped_reward(board_before, board_after_pair, dqn_player, terminal, winner):
    """
    Reward for one (DQN-acts → opponent-acts) pair.
    Terminal transitions use only the win/loss/draw signal.
    Non-terminal transitions get a small shaped bonus to speed up learning.
    """
    if terminal:
        if winner == dqn_player:    return R_WIN
        if winner == -dqn_player:   return R_LOSS
        return R_DRAW

    r = 0.0
    my_t_before  = _count_open_threes(board_before,     dqn_player)
    my_t_after   = _count_open_threes(board_after_pair, dqn_player)
    opp_t_before = _count_open_threes(board_before,    -dqn_player)
    opp_t_after  = _count_open_threes(board_after_pair,-dqn_player)
    if my_t_after  > my_t_before:  r += R_THREAT * (my_t_after  - my_t_before)
    if opp_t_after < opp_t_before: r += R_BLOCK  * (opp_t_before - opp_t_after)
    return r


# ── Core game loop ────────────────────────────────────────────────────

def play_one_game(online_net, opponent_fn, epsilon):
    """
    Play one full game; return a list of (s, a, r, s', done) experiences.

    MDP formulation: each 'step' for the DQN covers
      1. DQN chooses a column (ε-greedy).
      2. The opponent responds.
    This collapses the two-player game into a single-agent MDP.

    v1 BUG FIXED: experiences are appended for EVERY DQN decision,
    not only on the terminal transition. A typical game produces ~15
    experiences instead of 1, giving 15× more data per game.
    """
    board    = np.zeros((6, 7), dtype=np.int8)
    dqn_side = random.choice([+1, -1])
    opp_side = -dqn_side

    # Random warm-up: 0–6 random plies to diversify starting positions
    cur = +1
    for _ in range(random.randint(0, 6)):
        mvs = legal_cols(board)
        if not mvs: return []
        board = drop_piece(board, random.choice(mvs), cur)
        if wins(board, cur): return []  # warmup ended in a win — discard
        cur = -cur

    # If it's the opponent's turn after warmup, let them move first
    if cur == opp_side:
        mvs = legal_cols(board)
        if not mvs: return []
        board = drop_piece(board, opponent_fn(board, opp_side), opp_side)
        if wins(board, opp_side): return []

    exps = []

    while True:
        mvs = legal_cols(board)
        if not mvs:
            break  # draw; no DQN action this step (board was already full)

        state        = encode(board, dqn_side)
        board_before = board.copy()

        # ── DQN action (ε-greedy) ──────────────────────────────────
        if random.random() < epsilon:
            action = random.choice(mvs)
        else:
            q    = online_net(state[np.newaxis], training=False).numpy()[0]
            mask = np.full(7, -np.inf)
            mask[mvs] = q[mvs]
            action = int(np.argmax(mask))

        board = drop_piece(board, action, dqn_side)

        if wins(board, dqn_side):
            exps.append((state, action, R_WIN, np.zeros_like(state), True))
            break
        if not legal_cols(board):
            exps.append((state, action, R_DRAW, np.zeros_like(state), True))
            break

        # ── Opponent action ────────────────────────────────────────
        board = drop_piece(board, opponent_fn(board, opp_side), opp_side)

        if wins(board, opp_side):
            exps.append((state, action, R_LOSS, np.zeros_like(state), True))
            break
        if not legal_cols(board):
            exps.append((state, action, R_DRAW, np.zeros_like(state), True))
            break

        # ── Non-terminal: shaped intermediate reward ───────────────
        r          = shaped_reward(board_before, board, dqn_side, False, None)
        next_state = encode(board, dqn_side)
        exps.append((state, action, r, next_state, False))

    return exps


print('Game engine and reward shaping ready.')

In [ ]:
# Cell 6: @tf.function Training Step
#
# @tf.function traces Python into a TF computation graph, eliminating
# per-call Python overhead. This alone turns the 28 s/episode wall-time
# in v1 into <2 s/episode on a T4 GPU (~15× speedup).
#
# Double DQN (van Hasselt et al. 2016): use the online network to SELECT
# the best next action, but the slower-moving target network to EVALUATE it.
# Prevents Q-value overestimation caused by argmax noise.

@tf.function
def train_step(S, A, R, S2, D):
    """One Double-DQN Huber-loss gradient update."""
    with tf.GradientTape() as tape:
        # Q(s, a) for the action actually taken
        q_cur    = online_net(S, training=True)
        idx      = tf.stack([tf.range(tf.shape(A)[0]), A], axis=1)
        q_taken  = tf.gather_nd(q_cur, idx)

        # Double DQN: online selects, target evaluates
        q_next_on  = online_net(S2, training=False)
        best_a     = tf.cast(tf.argmax(q_next_on, axis=1), tf.int32)
        q_next_tgt = target_net(S2, training=False)
        idx2       = tf.stack([tf.range(tf.shape(best_a)[0]), best_a], axis=1)
        q_next_val = tf.gather_nd(q_next_tgt, idx2)

        # Bellman target; (1-D) masks terminal transitions
        target = tf.stop_gradient(R + GAMMA * q_next_val * (1.0 - D))

        # Huber loss is less sensitive to large TD errors than MSE
        loss = tf.reduce_mean(tf.keras.losses.Huber()(target, q_taken))

    grads = tape.gradient(loss, online_net.trainable_variables)
    optimizer.apply_gradients(zip(grads, online_net.trainable_variables))
    return loss


# Pre-compile tf.function with a real-shaped dummy batch
# (avoids a slow compile hit on the first training episode)
_dummy_s  = tf.zeros((BATCH_SIZE, 6, 7, 2))
_dummy_a  = tf.zeros((BATCH_SIZE,), dtype=tf.int32)
_dummy_r  = tf.zeros((BATCH_SIZE,))
_dummy_d  = tf.zeros((BATCH_SIZE,))
train_step(_dummy_s, _dummy_a, _dummy_r, _dummy_s, _dummy_d)
print('train_step compiled on GPU ✓')

In [ ]:
# Cell 7: Self-Play Opponent Pool
#
# Self-play (Silver et al., AlphaGo Zero 2017) is the single highest-impact
# technique for board-game RL. Training against snapshots of itself creates
# a natural curriculum: as the DQN improves, so do its opponents.
#
# The pool also includes random and heuristic opponents so the agent
# maintains robustness against non-neural strategies.

class SelfPlayPool:
    """Pool of frozen weight snapshots of the online network."""

    def __init__(self, capacity):
        self.capacity   = capacity
        self._weights   = []              # list of weight tensors
        self._frozen    = build_dueling_dqn(name='frozen')  # single reused model

    def add(self, model):
        """Save a snapshot of model's current weights."""
        self._weights.append([w.numpy() for w in model.trainable_variables])
        if len(self._weights) > self.capacity:
            self._weights.pop(0)

    def get_fn(self):
        """Return an opponent function backed by a random snapshot."""
        if not self._weights:
            return None
        ws = random.choice(self._weights)
        for var, w in zip(self._frozen.trainable_variables, ws):
            var.assign(w)
        frozen = self._frozen  # close over current weights
        def _fn(board, player):
            s    = encode(board, player)
            q    = frozen(s[np.newaxis], training=False).numpy()[0]
            mvs  = legal_cols(board)
            mask = np.full(7, -np.inf)
            mask[mvs] = q[mvs]
            return int(np.argmax(mask))
        return _fn

    def __len__(self):
        return len(self._weights)


# ── Baseline opponent functions ──────────────────────────────────────

def random_opp(board, player):
    """Pure random — simplest baseline."""
    return random.choice(legal_cols(board))

def heuristic_opp(board, player):
    """
    1-ply heuristic: take immediate win → block opponent win → prefer center.
    Stronger than random; forces the DQN to learn defensive play.
    """
    mvs = legal_cols(board)
    for c in mvs:
        if wins(drop_piece(board, c, player), player):
            return c
    for c in mvs:
        if wins(drop_piece(board, c, -player), -player):
            return c
    for c in [3, 2, 4, 1, 5, 0, 6]:  # center-first preference
        if c in mvs:
            return c
    return random.choice(mvs)


sp_pool = SelfPlayPool(capacity=SELFPLAY_CAP)

def choose_opponent_fn():
    """Mix: SELFPLAY_FRAC chance of self-play; rest split between heuristic/random."""
    if len(sp_pool) > 0 and random.random() < SELFPLAY_FRAC:
        fn = sp_pool.get_fn()
        if fn is not None:
            return fn
    return heuristic_opp if random.random() < 0.7 else random_opp


print(f'Self-play pool ready (cap={SELFPLAY_CAP})')
print('Baseline opponents: random + 1-ply heuristic')

In [ ]:
# Cell 8: Main Training Loop

print('Starting Dueling Double DQN v2 Training')
print('=' * 65)
print(f'  {NUM_EPISODES} episodes × {GAMES_PER_EP} games = {TOTAL_GAMES:,} total games')
print(f'  GPU: {"yes ✓" if tf.config.list_physical_devices("GPU") else "CPU only"}')
print('=' * 65)

log = {k: [] for k in ['episode', 'win_rate', 'loss', 'epsilon',
                        'buffer_size', 'ep_time', 'sp_pool_size']}

epsilon      = EPS_START
total_steps  = 0
recent_wins  = []      # sliding window of last 200 game outcomes
best_wr      = 0.0
ckpt_dir     = REPO_DIR / 'checkpoints'
log_path     = REPO_DIR / 'logs' / 'dqn_v2_log.json'
train_start  = time.time()

for ep in range(NUM_EPISODES):
    ep_start      = time.time()
    wins_this_ep  = 0
    ep_losses     = []

    # ── Collect games ──────────────────────────────────────────────
    opp_fn = choose_opponent_fn()

    for _ in range(GAMES_PER_EP):
        exps = play_one_game(online_net, opp_fn, epsilon)
        won  = any(e[2] == R_WIN for e in exps)
        wins_this_ep += int(won)
        recent_wins.append(int(won))
        if len(recent_wins) > 200:
            recent_wins.pop(0)
        for (s, a, r, s2, d) in exps:
            buf.push(s, a, r, s2, d)

    # ── Gradient updates ───────────────────────────────────────────
    if ep >= WARMUP_EPS and len(buf) >= BATCH_SIZE:
        n_updates = GAMES_PER_EP * UPDATES_PER_GAME
        for _ in range(n_updates):
            loss = train_step(*buf.sample(BATCH_SIZE))
            ep_losses.append(float(loss.numpy()))
            total_steps += 1

            # Sync target network periodically
            if total_steps % TARGET_SYNC_N == 0:
                target_net.set_weights(online_net.get_weights())

    # ── Epsilon decay ──────────────────────────────────────────────
    epsilon = max(EPS_END, epsilon * EPS_DECAY)

    # ── Self-play snapshot ─────────────────────────────────────────
    if (ep + 1) % SELFPLAY_FREQ == 0:
        sp_pool.add(online_net)

    # ── Checkpoint ────────────────────────────────────────────────
    if (ep + 1) % CKPT_EVERY == 0:
        online_net.save_weights(str(ckpt_dir / f'dqn_v2_ep{ep+1:04d}.h5'))

    # ── Logging ───────────────────────────────────────────────────
    wr      = wins_this_ep / GAMES_PER_EP
    avg_loss = float(np.mean(ep_losses)) if ep_losses else 0.0
    ep_time = time.time() - ep_start
    recent_wr = float(np.mean(recent_wins)) if recent_wins else 0.0

    if wr > best_wr:
        best_wr = wr
        online_net.save_weights(str(ckpt_dir / 'dqn_v2_best.h5'))

    log['episode'].append(ep)
    log['win_rate'].append(wr)
    log['loss'].append(avg_loss)
    log['epsilon'].append(float(epsilon))
    log['buffer_size'].append(len(buf))
    log['ep_time'].append(ep_time)
    log['sp_pool_size'].append(len(sp_pool))

    if ep % 10 == 0 or ep == NUM_EPISODES - 1:
        elapsed = time.time() - train_start
        eta = (elapsed / (ep + 1)) * (NUM_EPISODES - ep - 1)
        print(
            f'Ep {ep:>4d} | wr={wr:.2f} (recent={recent_wr:.2f}) | '
            f'loss={avg_loss:.4f} | ε={epsilon:.3f} | '
            f'buf={len(buf):>7,} | sp={len(sp_pool)} | '
            f'{ep_time:.1f}s | ETA {eta/60:.0f}m'
        )

# ── Save final model + log ────────────────────────────────────────
online_net.save_weights(str(ckpt_dir / 'dqn_v2_final.h5'))
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2)

elapsed = time.time() - train_start
print(f'\nTraining complete in {elapsed/60:.1f} min')
print(f'Final model: {ckpt_dir}/dqn_v2_final.h5')

In [ ]:
# Cell 9: Evaluation Against Fixed Opponents
#
# Use greedy play (ε=0) against the same fixed opponents every time
# so results are comparable across runs and to the PG model baseline.

def evaluate(model, opponent_fn, n_games=EVAL_GAMES, verbose=False):
    """Win rate of model (greedy, no exploration) against a fixed opponent."""
    wins_count = 0
    for i in range(n_games):
        exps = play_one_game(model, opponent_fn, epsilon=0.0)
        if any(e[2] == R_WIN for e in exps):
            wins_count += 1
    wr = wins_count / n_games
    if verbose:
        print(f'  {wins_count}/{n_games} = {wr:.1%}')
    return wr


print('Final evaluation (greedy, ε = 0):')
print('-' * 45)

wr_rand = evaluate(online_net, random_opp,    verbose=True)
wr_heur = evaluate(online_net, heuristic_opp, verbose=True)
avg_wr  = (wr_rand + wr_heur) / 2

print(f'\nvs random:        {wr_rand:.1%}')
print(f'vs heuristic:     {wr_heur:.1%}')
print(f'Average:          {avg_wr:.1%}')

# Evaluate vs a self-play snapshot if available
wr_self = None
if len(sp_pool) > 0:
    sp_fn  = sp_pool.get_fn()
    wr_self = evaluate(online_net, sp_fn, verbose=True)
    print(f'vs self-play:     {wr_self:.1%}')

# Compare to PG baseline
PG_BASELINE = 0.608
delta = avg_wr - PG_BASELINE
print(f'\nPG model baseline: {PG_BASELINE:.1%}')
print(f'DQN v2 average:    {avg_wr:.1%}  ({delta:+.1%} vs PG)')
print('Result:', 'BEATS PG ✓' if delta > 0 else 'Below PG — consider more training')

In [ ]:
# Cell 10: Training Curves and Summary

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Dueling Double DQN v2 — Training Analysis', fontsize=15, fontweight='bold')

episodes = log['episode']
W = 20  # smoothing window

def smooth(arr, w):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

# Win rate
ax = axes[0, 0]
ax.plot(episodes, log['win_rate'], alpha=0.3, color='steelblue')
ax.plot(episodes[W-1:], smooth(log['win_rate'], W), color='steelblue', lw=2, label=f'{W}-ep avg')
ax.axhline(PG_BASELINE, color='red', ls='--', label=f'PG {PG_BASELINE:.0%}')
ax.set_title('Win Rate'); ax.set_xlabel('Episode'); ax.set_ylabel('Win Rate')
ax.legend(); ax.grid(alpha=0.3)

# Loss
ax = axes[0, 1]
ax.plot(episodes, log['loss'], color='orange', lw=1.5)
ax.set_title('Huber Loss'); ax.set_xlabel('Episode'); ax.set_ylabel('Loss')
ax.grid(alpha=0.3)

# Epsilon decay
ax = axes[0, 2]
ax.plot(episodes, log['epsilon'], color='green', lw=2)
ax.axhline(EPS_END, color='gray', ls='--', label=f'floor {EPS_END}')
ax.set_title('Epsilon Decay'); ax.set_xlabel('Episode'); ax.set_ylabel('ε')
ax.legend(); ax.grid(alpha=0.3)

# Buffer fill
ax = axes[1, 0]
ax.plot(episodes, log['buffer_size'], color='purple', lw=2)
ax.axhline(BUFFER_CAP, color='gray', ls='--', label='capacity')
ax.set_title('Replay Buffer Size'); ax.set_xlabel('Episode'); ax.set_ylabel('Transitions')
ax.legend(); ax.grid(alpha=0.3)

# Episode time
ax = axes[1, 1]
ax.plot(episodes, log['ep_time'], color='brown', lw=1.5)
avg_t = np.mean(log['ep_time'])
ax.axhline(avg_t, color='gray', ls='--', label=f'avg {avg_t:.1f}s')
ax.set_title('Episode Wall Time'); ax.set_xlabel('Episode'); ax.set_ylabel('Seconds')
ax.legend(); ax.grid(alpha=0.3)

# Bar chart: DQN v2 vs PG
ax = axes[1, 2]
labels  = ['vs Random', 'vs Heuristic', 'DQN Avg', 'PG Baseline']
values  = [wr_rand, wr_heur, avg_wr, PG_BASELINE]
colors  = ['steelblue', 'steelblue', 'navy', 'red']
bars    = ax.bar(labels, values, color=colors, alpha=0.75)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.1%}',
            ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title('Win Rate Comparison'); ax.set_ylabel('Win Rate')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig_path = REPO_DIR / 'logs' / 'dqn_v2_curves.png'
plt.savefig(str(fig_path), dpi=120, bbox_inches='tight')
plt.show()

# ── Text summary ──────────────────────────────────────────────────
print('\n' + '=' * 58)
print('  DUELING DOUBLE DQN v2 — FINAL SUMMARY')
print('=' * 58)
print(f'  Total games        : {TOTAL_GAMES:,}')
print(f'  Buffer transitions : {len(buf):,} / {BUFFER_CAP:,}')
print(f'  Gradient steps     : {total_steps:,}')
print(f'  Training time      : {elapsed/60:.1f} min')
print(f'  Final ε            : {epsilon:.4f}')
print(f'  Self-play pool     : {len(sp_pool)} snapshots')
print()
print(f'  Win vs random      : {wr_rand:.1%}')
print(f'  Win vs heuristic   : {wr_heur:.1%}')
if wr_self is not None:
    print(f'  Win vs self-play   : {wr_self:.1%}')
print(f'  Average            : {avg_wr:.1%}')
print(f'  PG baseline        : {PG_BASELINE:.1%}')
print(f'  vs PG              : {delta:+.1%} ({"BEATS" if delta > 0 else "BELOW"} PG)')
print('=' * 58)